> * Name: <span style="color:red"> Rwan Ashraf  </span>

> * ID: <span style="color:red"> 221001757 </span>

> * Lab: <span style="color:red">1b </span>





# **Tasks**

## 1- Data loading:


In [2]:
import pandas as pd
# Trying 'utf-8' with error handling or alternative encodings for Arabic text
try:
    df = pd.read_csv('ar_reviews_100k.tsv', sep='\t', encoding='utf-8')
except UnicodeDecodeError:
    # Fallback to windows-1256 if utf-8 fails
    df = pd.read_csv('ar_reviews_100k.tsv', sep='\t', encoding='windows-1256')

df.head()

,label,text
0,Positive,ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...
1,Positive,أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...
2,Positive,هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...
3,Positive,خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...
4,Positive,ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...


## 2- Train Word2Vec to perform word embedding on the given dataset.

In [4]:
!pip install pyarabic
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyarabic.araby as araby
from pyarabic.araby import strip_tashkeel, strip_tatweel
from pyarabic.araby import tokenize as araby_tokenize
import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 6.1 MB/s eta 0:00:00


In [5]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 31.2 MB/s eta 0:00:00


In [6]:
from gensim.models import Word2Vec

In [7]:
def normalize(sentence):
    # re.sub(pattern (old), replacement (new), string) -> English
    # re.sub(new, old, string) -> Arabic, except for []

    # if the pattern is not found, the text is returned unchanged
    sentence = re.sub("[إأآا]", "ا", sentence)
    sentence = re.sub("ى", "ي", sentence)
    sentence = re.sub("ؤ", "ء", sentence)
    sentence = re.sub("ئ", "ء", sentence)
    sentence = re.sub("ة", "ه", sentence)
    return sentence

In [8]:
def clean_arabic_text(text):
    if not isinstance(text, str):
        return ""

    text = araby.strip_tashkeel(text)
    text = normalize(text)
    text = re.sub(r"[^\w\s]", "", text)
    return text

df['cleaned_text'] = df["text"].apply(clean_arabic_text)

df[['cleaned_text']].head()

,cleaned_text
0,ممتاز نوعا ما النظافه والموقع والتجهيز والشاط...
1,احد اسباب نجاح الامارات ان كل شخص في هذه الدول...
2,هادفه وقويه تنقلك من صخب شوارع القاهره الي هد...
3,خلصنا مبدءيا اللي مستني ابهار زي الفيل الازرق...
4,ياسات جلوريا جزء لا يتجزا من دبي فندق متكامل ...


In [ ]:
import string
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

def tokenize_text(text):
    # Remove punctuation and tokenize
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    # Optional: remove very short tokens or stopwords
    tokens = [word for word in tokens if len(word) > 1]
    return tokens

# Apply tokenization
df['tokens'] = df['cleaned_text'].apply(tokenize_text)

# Now create the corpus for Word2Vec: list of lists
sentences = df['tokens'].tolist()

print(sentences[:3])

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


[['ممتاز', 'نوعا', 'ما', 'النظافه', 'والموقع', 'والتجهيز', 'والشاطيء', 'المطعم'], ['احد', 'اسباب', 'نجاح', 'الامارات', 'ان', 'كل', 'شخص', 'في', 'هذه', 'الدوله', 'يعشق', 'ترابها', 'نحن', 'نحب', 'الامارات', 'ومضات', 'من', 'فكر', 'نصاءح', 'لدوله', 'تطمح', 'بالصفوف', 'الاولي', 'قاءد', 'لا', 'يقبل', 'الا', 'براحه', 'شعبه', 'وتوفر', 'كل', 'سب', 'العيش', 'الكريم', 'حكم', 'مواقف', 'ونصاءح', 'لكل', 'فرد', 'فينا', 'ليس', 'بمجرد', 'كتاب', 'سياسي', 'كما', 'كنت', 'اعتقد', 'يستحق', 'القراءه', 'مرات', 'كثيره'], ['هادفه', 'وقويه', 'تنقلك', 'من', 'صخب', 'شوارع', 'القاهره', 'الي', 'هدوء', 'جبال', 'الشيشان', 'للتعرف', 'علي', 'حقيقه', 'ما', 'يجري', 'في', 'تلك', 'البلاد', 'من', 'حروب', 'ضاربه', 'بحق', 'المسلمين', 'جزء', 'كبير', 'من', 'تاريخ', 'تلك', 'المنطقه', 'التضحيه', 'الرجوله', 'الوفاء', 'والكثير', 'من', 'القيم', 'الاخري', 'اثبتت', 'وجودها', 'في', 'تلك', 'الروايه', 'البسيطه']]


In [ ]:
from gensim.models import Word2Vec
w2v = Word2Vec(sentences, vector_size=100, window=5, workers=4, epochs=10, min_count=5)


In [ ]:
w2v.wv['ممتاز']

array([ 1.5403234e+00, -9.3108487e-01, -2.6001530e+00, -1.4016281e+00,
       -2.5726576e+00,  5.7304651e-02, -1.1171486e+00,  1.2822554e+00,
        1.7048765e+00,  7.3664719e-01, -1.9745096e+00,  6.6896343e-01,
       -3.5038772e+00,  1.8976031e-01,  3.3165210e-01,  3.3449020e+00,
        8.3642817e-01, -3.4168750e-01, -6.0575187e-01, -2.7004573e-01,
       -1.3181583e+00,  8.8887501e-01,  8.8183844e-01, -2.2796314e+00,
        2.5530747e-01,  3.1320006e-01,  9.9071515e-01, -3.9566392e-01,
       -2.4628611e-01,  1.9901011e+00,  2.5823693e+00,  3.4874138e-01,
       -4.7553626e-01, -9.3253940e-02, -2.0339243e+00, -2.4108746e+00,
       -2.5690880e-01, -5.8294043e-02,  2.5668826e+00, -1.7422669e+00,
        2.6835203e-01,  9.9110901e-01,  1.0163568e+00,  9.5926023e-01,
        2.6856797e+00,  2.2978399e+00, -2.5661701e-01,  5.6698847e-01,
       -1.7225393e+00, -6.2280172e-01, -1.4244738e+00,  1.3279363e+00,
        4.7869366e-01,  1.0271250e+00, -1.0621064e+00,  6.9488031e-01,
      

In [ ]:
w2v.wv["rwan"]

KeyError: "Key 'rwan' not present"

## 3- Use AraVec to perform word embedding on the given dataset.
Note: You can use this version: full_grams_cbow_100_twitter.mdl




In [ ]:
import kagglehub

path = kagglehub.dataset_download("hadeerkhalednabil/full-grams-cbow-100-twitter")
print(path)

100%|██████████| 1.05G/1.05G [00:08<00:00, 129MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/hadeerkhalednabil/full-grams-cbow-100-twitter/versions/2


In [ ]:
from gensim.models import Word2Vec as W2V
import os

file_path = os.path.join(path, "full_grams_cbow_100_twitter.mdl")
# Load the pre-trained AraVec model
aravec_model = W2V.load(file_path)
print("AraVec model loaded successfully!")

AraVec model loaded successfully!


In [ ]:
def get_word_embedding_aravec(word, model):
    """Get embedding for a single word using AraVec"""
    try:
        return model.wv[word]
    except KeyError:
        return None

def get_sentence_embedding_aravec(text, model, method='average'):
    """
    Get sentence embedding using AraVec
    methods: 'average', 'sum'
    """
    if pd.isna(text) or not isinstance(text, str):
        return np.zeros(model.vector_size)

    words = text.split()
    vectors = []

    for word in words:
        vec = get_word_embedding_aravec(word, model)
        if vec is not None:
            vectors.append(vec)

    if not vectors:
        return np.zeros(model.vector_size)

    if method == 'average':
        return np.mean(vectors, axis=0)
    elif method == 'sum':
        return np.sum(vectors, axis=0)

    return np.mean(vectors, axis=0)

print(f"Using AraVec model with {aravec_model.vector_size} dimensions")

Using AraVec model with 100 dimensions


In [ ]:
print("Creating AraVec embeddings for all reviews...")
df['aravec_embedding'] = df['cleaned_text'].apply(
    lambda x: get_sentence_embedding_aravec(x, aravec_model)
)

# Convert to numpy array
X_aravec = np.vstack(df['aravec_embedding'].values)
print(f"✅ AraVec embeddings created! Shape: {X_aravec.shape}")


Creating AraVec embeddings for all reviews...
✅ AraVec embeddings created! Shape: (99999, 100)


In [ ]:
# Test some words with AraVec
test_words_aravec = ['روان','ممتاز', 'سيء', 'جميل','برجر', 'كتاب']

print("\nAraVec Word Embeddings Examples:")
print("-" * 60)

for word in test_words_aravec:
    embedding = get_word_embedding_aravec(word, aravec_model)
    if embedding is not None:
        print(f"\n'{word}':")
        print(f"  Vector shape: {embedding.shape}")
        print(f"  First 10 dimensions: {embedding[:10]}")
        print(f"  Vector norm: {np.linalg.norm(embedding):.4f}")

        # Find similar words
        try:
            similar = aravec_model.wv.most_similar(word, topn=5)
            print(f"  Most similar words:")
            for sim_word, score in similar:
                print(f"    {sim_word}: {score:.4f}")
        except KeyError:
            print("  Similarity function not available for this word or model")
    else:
        print(f"\n'{word}': Not found in AraVec vocabulary.")

# Test out-of-vocabulary words (AraVec, like Word2Vec, generally cannot handle OOV words unless explicitly handled in the get_embedding function)
oov_words_aravec = ['كلمةغيرموجودة', 'جديدبالكامل']
print("\n" + "="*60)
print("Testing Out-of-Vocabulary Words (AraVec):")
print("-" * 60)

for word in oov_words_aravec:
    embedding = get_word_embedding_aravec(word, aravec_model)
    if embedding is not None:
        print(f"\n'{word}': Has embedding: YES")
    else:
        print(f"\n'{word}': No embedding (OOV)")


AraVec Word Embeddings Examples:
------------------------------------------------------------

'روان':
  Vector shape: (100,)
  First 10 dimensions: [-0.5554924   1.332018   -1.5532793  -0.92974347 -1.9206158   0.97820693
  0.20642827 -0.5581675  -1.1805456  -1.7522584 ]
  Vector norm: 13.1372
  Most similar words:
    رهف: 0.8866
    مها: 0.8680
    رزان: 0.8638
    نوف: 0.8636
    مرام: 0.8598

'ممتاز':
  Vector shape: (100,)
  First 10 dimensions: [-1.6708798  -1.3976244   1.2246977   0.9495364   1.4465071  -0.39980486
 -0.10589279  0.7142365  -0.23952107 -2.709499  ]
  Vector norm: 18.6242
  Most similar words:
    جيد: 0.8539
    ممتااز: 0.8204
    مميز: 0.7737
    وممتاز: 0.7732
    ممتازه: 0.7486

'سيء':
  Vector shape: (100,)
  First 10 dimensions: [-2.008132    0.74440926  2.2431943  -0.55062586  1.5909983   3.299007
  0.65260476  0.10069178  0.05515162 -0.55573934]
  Vector norm: 19.4933
  Most similar words:
    سيئ: 0.9363
    سئ: 0.8661
    جيد: 0.8244
    سيء_جدا: 0.8159

## 4- Compare the performance of (3) and (4) on a simple text classification task using logistic regression. Comment on the results.
Note: Use the same code used in the lab for logistic regression.


In [ ]:
import re
import random
import numpy as np
from typing import List
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import numpy as np

# 1) Helper function to get sentence vectors (averaging word vectors)
def get_sentence_vec(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# 2) Prepare data and labels
y = df['label'].values
SEED = 42

# Create feature matrices
print("Generating feature matrices...")
X_w2v_all = np.vstack([get_sentence_vec(t, w2v) for t in df['tokens']])
X_aravec_all = np.vstack([get_sentence_vec(t.split(), aravec_model) for t in df['cleaned_text']])

# 3) Split for Word2Vec
X_train_w2v, X_test_w2v, y_train, y_test = train_test_split(
    X_w2v_all, y, test_size=0.2, random_state=SEED, stratify=y
)

# 4) Train/Predict Word2Vec
clf_w2v = LogisticRegression(max_iter=2000, random_state=SEED)
clf_w2v.fit(X_train_w2v, y_train)
pred_w2v = clf_w2v.predict(X_test_w2v)

# 5) Split for AraVec
X_train_aravec, X_test_aravec, _, _ = train_test_split(
    X_aravec_all, y, test_size=0.2, random_state=SEED, stratify=y
)

# 6) Train/Predict AraVec
clf_aravec = LogisticRegression(max_iter=2000, random_state=SEED)
clf_aravec.fit(X_train_aravec, y_train)
pred_aravec = clf_aravec.predict(X_test_aravec)

# 7) Results Comparison
print("=== Custom Word2Vec Performance ===")
print("Accuracy:", round(accuracy_score(y_test, pred_w2v), 4))
print("\nClassification report:\n", classification_report(y_test, pred_w2v))

print("\n=== Pre-trained AraVec Performance ===")
print("Accuracy:", round(accuracy_score(y_test, pred_aravec), 4))
print("\nClassification report:\n", classification_report(y_test, pred_aravec))

Generating feature matrices...
=== Custom Word2Vec Performance ===
Accuracy: 0.5823

Classification report:
               precision    recall  f1-score   support

       Mixed       0.50      0.48      0.49      6666
    Negative       0.61      0.66      0.63      6667
    Positive       0.63      0.61      0.62      6667

    accuracy                           0.58     20000
   macro avg       0.58      0.58      0.58     20000
weighted avg       0.58      0.58      0.58     20000


=== Pre-trained AraVec Performance ===
Accuracy: 0.5699

Classification report:
               precision    recall  f1-score   support

       Mixed       0.50      0.47      0.48      6666
    Negative       0.59      0.64      0.62      6667
    Positive       0.61      0.60      0.60      6667

    accuracy                           0.57     20000
   macro avg       0.57      0.57      0.57     20000
weighted avg       0.57      0.57      0.57     20000



###Comment on the result
aravec and word2vec seems that both of them have a similar accuracy

## 5- Train FastText to perform word embedding on the given dataset.


In [ ]:
from gensim.models import FastText

ft = FastText(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,              # 1 = skip-gram, 0 = CBOW
    epochs=30,
    min_n=3, max_n=6   # <-- underscores
)

In [ ]:
print("cos(سيء, جيد) =", round(ft.wv.similarity("king","queen"), 3))
print("OOV vector shape:", ft.wv.get_vector("جيد").shape)  # OOV via subwords

cos(سيء, جيد) = -0.202
OOV vector shape: (100,)


## 6- Use FastText to perform word embedding on the given dataset.
Note: You can use this version: fasttext-pretrained-arabic-word-vectors




In [10]:
import kagglehub
# Download the specific pre-trained FastText model
ft_pretrained_path = kagglehub.dataset_download("javadhelali/fasttext-pretrained-arabic-word-vectors")
print("Path to pre-trained FastText:", ft_pretrained_path)

100%|██████████| 1.20G/1.20G [00:11<00:00, 113MB/s]

Extracting files...


Path to pre-trained FastText: /root/.cache/kagglehub/datasets/javadhelali/fasttext-pretrained-arabic-word-vectors/versions/1


In [11]:
from gensim.models import KeyedVectors
import os

# The available file is 'cc.ar.300.vec'
model_file = os.path.join(ft_pretrained_path, 'cc.ar.300.vec')

print(f"Attempting to load: {model_file}")

try:
    # Use load_word2vec_format for .vec files
    ft_model = KeyedVectors.load_word2vec_format(model_file, binary=False)
    print("Pre-trained FastText (.vec) loaded successfully!")
except Exception as e:
    print(f"Error: {e}")

Attempting to load: /root/.cache/kagglehub/datasets/javadhelali/fasttext-pretrained-arabic-word-vectors/versions/1/cc.ar.300.vec
Pre-trained FastText (.vec) loaded successfully!


In [14]:
def get_sentence_embedding_ft(text, model):
    if pd.isna(text) or not isinstance(text, str):
        return np.zeros(model.vector_size)

    words = text.split()
    # For KeyedVectors, we access the model directly, not via .wv
    vectors = [model[word] for word in words if word in model]

    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

print("Generating FastText embeddings for the dataset...")
df['fasttext_embedding'] = df['cleaned_text'].apply(lambda x: get_sentence_embedding_ft(x, ft_model))

X_fasttext = np.vstack(df['fasttext_embedding'].values)
print(f"FastText embeddings created. Shape: {X_fasttext.shape}")

Generating FastText embeddings for the dataset...
FastText embeddings created. Shape: (15672, 300)
